In [1]:
%pip install -r requirements-offline.txt --quiet

/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-29/.check-env/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Exercise — Module M4: เสียง → ตัวเลข → RF (ESC-50)

ทำในไฟล์นี้ — เติมโค้ดตรง `# TODO` แล้วตอบคำถามใน markdown ที่กำหนด
(เปิด `book.ipynb` คู่ไปด้วยจะง่ายขึ้น) รันเซลล์ตามลำดับจากบนลงล่าง

**ที่มา:** ESC-50 ([github.com/karoldvl/ESC-50](https://github.com/karoldvl/ESC-50))
— license **CC BY-NC 3.0 (non-commercial)** ห้ามแจกต่อเชิงพาณิชย์

## 0. เตรียมข้อมูล (ให้มาแล้ว — รันผ่าน)

หา ESC-50 บนเครื่อง (หรือดาวน์โหลดครั้งเดียว ~600 MB) → คำนวณ MFCC feature
40 ตัวเลขต่อคลิปทั้ง 2,000 คลิป → เทรน RF ด้วย 5-fold protocol ทางการแบบที่
book.ipynb ทำ — ใช้ผล `accs` นี้เป็น baseline ของทุกข้อ

In [2]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

from pathlib import Path  # noqa: E402


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root "
        "(uv run jupyter lab)"
    )


REPO = repo_root()


In [3]:
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import requests  # noqa: E402
import zipfile  # noqa: E402
import librosa  # noqa: E402

ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip"


def find_esc50() -> Path:
    """ค้นหาโฟลเดอร์ ESC-50 ที่แตกไว้แล้วจากตำแหน่งที่รู้จัก."""
    env_dir = os.environ.get("ESC50_DIR")
    candidates = [
        Path(env_dir) if env_dir else None,
        REPO / "datasets" / "esc50" / "ESC-50-master",
        Path.home() / ".cache" / "esc50" / "ESC-50-master",
        Path.home() / "Downloads" / "ESC-50-master",
    ]
    for c in candidates:
        if c is not None and (c / "meta" / "esc50.csv").exists():
            print("พบ ESC-50 ที่:", c)
            return c
    return None


ESC50 = find_esc50()
if ESC50 is None:
    cache = Path.home() / ".cache" / "esc50"
    cache.mkdir(parents=True, exist_ok=True)
    zpath = cache / "esc50-master.zip"
    if not zpath.exists():
        print(f"ดาวน์โหลด ESC-50 (~600 MB) จาก {ESC50_URL}")
        with requests.get(ESC50_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(cache)
    ESC50 = cache / "ESC-50-master"


พบ ESC-50 ที่: /Users/utarn/.cache/esc50/ESC-50-master


In [4]:
meta = pd.read_csv(ESC50 / "meta" / "esc50.csv")
id2cat = (
    meta.drop_duplicates("target")
    .sort_values("target")["category"]
    .tolist()
)
paths = [ESC50 / "audio" / f for f in meta["filename"]]
y_all = meta["target"].to_numpy()
folds = meta["fold"].to_numpy()
print("จำนวนคลิป:", len(meta), "| จำนวน class:",
      meta["category"].nunique())
print(meta["fold"].value_counts().sort_index().to_string())
print(meta["category"].value_counts().head(8).to_string())


จำนวนคลิป: 2000 | จำนวน class: 50
fold
1    400
2    400
3    400
4    400
5    400
category
dog                40
chirping_birds     40
vacuum_cleaner     40
thunderstorm       40
door_wood_knock    40
can_opening        40
crow               40
clapping           40


In [5]:
SR = 22050


def mfcc_row(path: Path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 40 ตัวเลข (mean+std ของ MFCC 20 ตัว)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)])


def evaluate_folds(Xa, ya, fa, n_est=200):
    """วน 5-fold ทางการของ ESC-50 — คืน dict {fold: accuracy}."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    out = {}
    for k in range(1, 6):
        tr, te = fa != k, fa == k
        clf = RandomForestClassifier(
            n_estimators=n_est, n_jobs=-1, random_state=SEED
        )
        clf.fit(Xa[tr], ya[tr])
        out[k] = accuracy_score(ya[te], clf.predict(Xa[te]))
        print(f"  fold {k}: accuracy = {out[k]:.4f}")
    print(f"  mean = {float(np.mean(list(out.values()))):.4f}")
    return out


from joblib import Parallel, delayed  # noqa: E402

X = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_row)(p) for p in paths
))
print("X:", X.shape)
accs = evaluate_folds(X, y_all, folds)

X: (2000, 40)


  fold 1: accuracy = 0.4350


  fold 2: accuracy = 0.4375


  fold 3: accuracy = 0.4600


  fold 4: accuracy = 0.4725


  fold 5: accuracy = 0.4675
  mean = 0.4545


## ข้อ 1 — เพิ่ม delta-MFCC: จับ "ความเปลี่ยนแปลง" เพิ่มไหม

feature เดิมสรุป MFCC ด้วย mean+std แต่ std วัดแค่ "กระจายแค่ไหน" ไม่ได้บอก
"เสียงวิ่งขึ้น/ลงอย่างไร" — **delta** (อนุพันธ์ตามเวลา) จับสิ่งนั้น
เติมโค้ดให้ได้ feature 80 ตัว (mean+std ของ MFCC และของ delta) แล้วเทียบ
accuracy กับ baseline

In [6]:
def mfcc_delta_row(path: Path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 80 ตัวเลข (MFCC + delta, mean+std)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    d = ...  # TODO 1: อนุพันธ์ตามเวลาของ m — librosa.feature.delta(m, axis=1)
    return np.concatenate([
        m.mean(axis=1), m.std(axis=1),
        d.mean(axis=1), d.std(axis=1),
    ])


X2 = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_delta_row)(p) for p in paths
))
print("X2:", X2.shape)
accs2 = evaluate_folds(X2, y_all, folds)

AttributeError: 'ellipsis' object has no attribute 'mean'

คำตอบข้อ 1: (accuracy เฉลี่ยเปลี่ยนจาก 0.4545 ไปเท่าไร — คุ้มไหมกับ
feature ที่เพิ่มเป็นสองเท่า)

## ข้อ 2 — ต้นไม้มากเท่าไรถึง "พอ"

RF ที่ให้มาใช้ `n_estimators=200` — ถ้าลดเหลือ 50 (เร็วขึ้น 4 เท่า) เสีย
accuracy ไปเท่าไร และถ้าเพิ่มเป็น 500 ได้คืนกี่จุด? (ใช้ `evaluate_folds(...,
n_est=...)`)

In [7]:
# TODO 2: รัน evaluate_folds ด้วย n_est=50 และ n_est=500
acc_50 = ...  # noqa: F841
acc_500 = ...  # noqa: F841

คำตอบข้อ 2: (50 → เท่าไร, 500 → เท่าไร, สรุปว่า 200 อยู่จุดไหนของ
ช่วงค่าตอบแทนต่อเวลา)

## ข้อ 3 — โมเดลสับสนคู่ไหน (แบบที่ "หูมนุษย์" ก็เข้าใจผิด)

จาก baseline (accs): ทำนาย fold 5 แล้วนับคู่สับสนบ่อยสุดด้วย
`Counter` — คู่พวกนี้สมเหตุสมผลไหมถ้าลอง "ฟังในหัว" ดู


In [8]:
from collections import Counter  # noqa: E402

te5 = folds == 5
# TODO 3: เทรน RF ด้วย 4 folds (ไม่รวม 5) บน X — แล้วทำนาย X[te5]
# ใบ้: ใช้ RandomForestClassifier แบบเดียวกับใน evaluate_folds
rf5 = ...  # noqa: F841
pred5 = ...  # noqa: F841

wrong = Counter(
    (id2cat[t], id2cat[p])
    for t, p in zip(y_all[te5], pred5) if t != p
)
print(wrong.most_common(8))

TypeError: 'ellipsis' object is not iterable

คำตอบข้อ 3: (คู่สับสนบ่อยสุด — เหตุผลเชิงเสียงที่ทำให้ MFCC mean+std
แยกไม่ออก)

## ข้อ 4 — โจทย์แคบลง = ง่ายขึ้นแค่ไหน (ESC-10)

ESC-50 มี subset 10 class ที่ถูกเลือกไว้ (`esc10 == True` — 400 คลิป)
จำแนก 10 class ง่ายกว่า 50 มากแค่ไหน? (กรอง mask แล้วเทรน RF ด้วย
`evaluate_folds` บน subset — รันเร็วมากเพราะมีแค่ 400 คลิป)

In [9]:
# TODO 4: mask = meta["esc10"].to_numpy() แล้วเทรนบน X[mask], y_all[mask],
# folds[mask]
mask = ...  # noqa: F841
accs10 = ...  # noqa: F841

คำตอบข้อ 4: (accuracy เฉลี่ยของ ESC-10 — สูงขึ้นกี่เท่าของโจทย์ 50 class
และการเดาสุ่มต่างกันอย่างไร)

## ข้อ 5 — ตัดสินใจแบบนักเมโทรโลยี (ตอบใน markdown)

โจทย์จริงเข้ามา: ทีมคุณอัดเสียงเครื่องจักรมา 200 คลิป (5 วินาที) แบ่ง
"ปกติ / ผิดปกติ" และเครื่องที่จะ deploy ไม่มี GPU — **เลือกวิธี: RF บน MFCC หรือ
spectrogram CNN?** พร้อมเหตุผล 2 ข้อที่ชี้เฉพาะโจทย์นี้ (ใบ้: ปริมาณข้อมูล,
เครื่องปลายทาง, ความสามารถในการตรวจสอบ และคำตอบจากข้อ 4)

คำตอบข้อ 5:



## สรุป (เติมหลังทำครบ)

- delta-MFCC ช่วย/ไม่ช่วย: ...
- จำนวนต้นไม้ที่ "พอ": ...
- ข้อจำกัดสำคัญของ feature แบบสรุปค่า (mean+std) คือ: ...
- สิ่งที่จะไปดูต่อใน module4_esc50_cnn คือ: ...